In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split, cross_val_score


In [ ]:

#this is path to the dataset and the dataset is getting loaded using pandas
path = "/content/spambase.data"
df = pd.read_csv(path, header=None)

In [ ]:

#these are all the columns from the spambase .names file
feat = ['word_freq_make', 'word_freq_address', 'word_freq_all', 'word_freq_3d',
    'word_freq_our', 'word_freq_over', 'word_freq_remove', 'word_freq_internet',
    'word_freq_order', 'word_freq_mail', 'word_freq_receive', 'word_freq_will',
    'word_freq_people', 'word_freq_report', 'word_freq_addresses', 'word_freq_free',
    'word_freq_business', 'word_freq_email', 'word_freq_you', 'word_freq_credit',
    'word_freq_your', 'word_freq_font', 'word_freq_000', 'word_freq_money',
    'word_freq_hp', 'word_freq_hpl', 'word_freq_george', 'word_freq_650',
    'word_freq_lab', 'word_freq_labs', 'word_freq_telnet', 'word_freq_857',
    'word_freq_data', 'word_freq_415', 'word_freq_85', 'word_freq_technology',
    'word_freq_1999', 'word_freq_parts', 'word_freq_pm', 'word_freq_direct',
    'word_freq_cs', 'word_freq_meeting', 'word_freq_original', 'word_freq_project',
    'word_freq_re', 'word_freq_edu', 'word_freq_table', 'word_freq_conference',
    'char_freq_;', 'char_freq_(', 'char_freq_[', 'char_freq_!', 'char_freq_$',
    'char_freq_#', 'capital_run_length_average', 'capital_run_length_longest',
    'capital_run_length_total', 'spam']

df.columns = feat
# we take all features in X and drop all columns except spam for y
X = df.drop('spam', axis=1).values
y = df['spam'].values

In [ ]:
# we split the data into 75 percent training and 25 percent testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)


# Here the scaler is getting initialized
s = StandardScaler()

#Here the Xtrain is getting scaled
X_train_after_scaler = s.fit_transform(X_train)

# Here I am adding a bias column
arr1 = np.ones((X_train_after_scaler.shape[0], 1))
X_train_with_bias = np.concatenate([arr1, X_train_after_scaler], axis=1)


#Here the Xtest is getting scaled
X_test_after_scaler = s.transform(X_test)

# Here I am adding a bias column
arr2 = np.ones((X_test_after_scaler.shape[0], 1))
X_test_with_bias = np.concatenate([arr2, X_test_after_scaler], axis=1)





In [ ]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def cross_entropy(y, h):
    h = np.clip(h, 1e-10, 1 - 1e-10)
    return -np.mean(y * np.log(h) + (1 - y) * np.log(1 - h))

In [ ]:
#list of learning rates
learningRates = [0.01, 0.1, 0.5]

#List of no of iterations
iterationLi = [10, 50, 100]

#no of training examples
n = X_train_with_bias.shape[0]

In [ ]:
#  here we take learning rate and do 10 iteration , 50 iteration and 100 iteration and see the loss for each case
# we are doing gradient descent here
print("cross-entropy loss objective at iteration 10, 50, 100 for different learning rate")
for l in learningRates:
    for i in iterationLi:
        theta = np.zeros(X_train_with_bias.shape[1])
        for j in range(i):
            h = sigmoid(X_train_with_bias @ theta)
            err = h - y_train
            grad = (1 / n) * (X_train_with_bias.T @ err)
            theta = theta - (l * grad)
        # we do cross entropy loss function with y true and predicted value
        loss = cross_entropy(y_train, sigmoid(X_train_with_bias @ theta))
        print(f"Alpha={l}, Iterations={i}, Loss={loss}")

cross-entropy loss objective at iteration 10, 50, 100 for different learning rate
Alpha=0.01, Iterations=10, Loss=0.6511669857747711
Alpha=0.01, Iterations=50, Loss=0.5416199336398835
Alpha=0.01, Iterations=100, Loss=0.4687340601821987
Alpha=0.1, Iterations=10, Loss=0.46537732773525686
Alpha=0.1, Iterations=50, Loss=0.32412438774423313
Alpha=0.1, Iterations=100, Loss=0.28907843871268457
Alpha=0.5, Iterations=10, Loss=0.32025067969930787
Alpha=0.5, Iterations=50, Loss=0.2588942577504775
Alpha=0.5, Iterations=100, Loss=0.24377756612458096


In [ ]:


print("\nMetrics at 100 iterations for testing dataset:")

for l in learningRates:
    theta = np.zeros(X_train_with_bias.shape[1])

    # Train for 100 iterations
    for j in range(100):
        h = sigmoid(X_train_with_bias @ theta)
        err = h - y_train
        grad = (1 / n) * (X_train_with_bias.T @ err)
        theta = theta - (l * grad)


    probs = sigmoid(X_test_with_bias @ theta)
    pred = []

    for p in probs:
        if p >= 0.5:
            pred.append(1)
        else:
            pred.append(0)

    pred = np.array(pred)


    acc = accuracy_score(y_test, pred)
    prec = precision_score(y_test, pred)
    rec = recall_score(y_test, pred)
    f1 = f1_score(y_test, pred)


    print("Learning Rate=" + str(l))
    print("accuracy=" + str(acc) +
          " precision=" + str(prec) +
          " recall=" + str(rec) +
          " f1=" + str(f1))


Metrics at 100 iterations for testing dataset:
Learning Rate=0.01
accuracy=0.9035621198957429 precision=0.9008810572687225 recall=0.8610526315789474 f1=0.8805166846071044
Learning Rate=0.1
accuracy=0.9061685490877498 precision=0.9237875288683602 recall=0.8421052631578947 f1=0.8810572687224669
Learning Rate=0.5
accuracy=0.9157254561251086 precision=0.9395348837209302 recall=0.8505263157894737 f1=0.8928176795580111


In [ ]:


# we are training a model
mod1 = LogisticRegression(max_iter=100)
mod1.fit(X_train, y_train)


# we are getting predictions on training data and checking metrics
pred = mod1.predict(X_train)

print("Training Set:")
print("accuracy =", accuracy_score(y_train, pred))
print("precision =", precision_score(y_train, pred))
print("recall =", recall_score(y_train, pred))
print("f1 =", f1_score(y_train, pred))


# now we are testing here and checking metrics on unseen data
pred = mod1.predict(X_test)

print("Testing Set:")
print("accuracy =", accuracy_score(y_test, pred))
print("precision =", precision_score(y_test, pred))
print("recall =", recall_score(y_test, pred))
print("f1 =", f1_score(y_test, pred))

Training Set:
accuracy = 0.9202898550724637
precision = 0.8993238166791886
recall = 0.8946188340807175
f1 = 0.8969651554889472
Testing Set:
accuracy = 0.9287576020851434
precision = 0.9337748344370861
recall = 0.8905263157894737
f1 = 0.9116379310344828


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
